In [69]:
import pandas as pd
import numpy as np
import re

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score


Load text file 


In [70]:
file_path="../data/cuad_contracts.txt"

with open(file_path,"r",encoding="utf-8") as f:
    raw_text= f.read()
    
len(raw_text)

26034

Split raw text into clauses 


In [71]:
clauses=re.split(
    r'\n\d+\.|\n[A-Z][A-Z\s]{3,}\n',
    raw_text
)

clauses=[c.strip() for c in clauses if len(c.strip())>150]

len(clauses)

14

Create Dataframe

In [72]:
df=pd.DataFrame({
    "text":clauses
})

df.head()

,text
0,Contract Understanding Atticus Dataset (CUAD) ...
1,"The files in CUAD v1 include 1 CSV file, 1 SQu..."
2,The labels correspond to 41 categories of lega...
3,Category (incl. context and answer)\n\tDescrip...
4,"The contracts were sourced from EDGAR, the Ele..."


ASSign risk lables 

In [73]:
def assign_risk(text):
    t=text.lower()
    if any(k in t for k in ["terminate", "liability", "indemnif", "penalty", "damages"]):
        return "High Risk"
    elif any(k in t for k in ["payment", "confidential", "warranty","governing law", "prior notice", "fees"]):
        return "Medium Risk"
    else:
        return "Low Risk"
    
    
df["label"]=df["text"].apply(assign_risk)
df.head()        

,text,label
0,Contract Understanding Atticus Dataset (CUAD) ...,Low Risk
1,"The files in CUAD v1 include 1 CSV file, 1 SQu...",Low Risk
2,The labels correspond to 41 categories of lega...,High Risk
3,Category (incl. context and answer)\n\tDescrip...,High Risk
4,"The contracts were sourced from EDGAR, the Ele...",Low Risk


manual data adding 

In [74]:
high_risk_clauses = [
    "The company may terminate this agreement at any time without prior notice.",
    "The user shall indemnify the company against all claims, losses, and damages.",
    "The company shall not be liable for any indirect or consequential damages.",
    "This agreement may be terminated unilaterally by the company.",
    "All legal costs shall be borne solely by the user.",
    "The company reserves the right to modify terms without user consent.",
    "The user waives any right to claim damages from the company.",
    "The company shall have unlimited liability protection.",
    "Termination may occur immediately upon breach.",
    "The user is responsible for all penalties arising from misuse."
]

df_high = pd.DataFrame({
    "text": high_risk_clauses,
    "label": "High Risk"
})


In [75]:
medium_risk_clauses = [
    "Payment must be made within thirty days of invoice date.",
    "Confidential information shall not be disclosed to third parties.",
    "The warranty period shall be valid for one year.",
    "The agreement shall be governed by the laws of India.",
    "Either party may terminate the agreement with prior notice.",
    "The company may revise service charges with prior notice.",
    "The user agrees to comply with company policies.",
    "Payment delays may result in service suspension.",
    "Confidentiality obligations survive termination.",
    "Limited liability applies under applicable law."
]

df_medium = pd.DataFrame({
    "text": medium_risk_clauses,
    "label": "Medium Risk"
})


In [76]:
low_risk_clauses = [
    "This agreement defines the terms and conditions of service.",
    "Definitions and interpretations are provided in this section.",
    "The headings are for convenience only.",
    "This document outlines general service information.",
    "This agreement is effective from the date of signing.",
    "All notices shall be in writing.",
    "The clauses are numbered for reference purposes.",
    "This section describes the scope of services.",
    "The agreement consists of standard provisions.",
    "General information is provided herein."
]

df_low = pd.DataFrame({
    "text": low_risk_clauses,
    "label": "Low Risk"
})


Merge+Shuffle dataset

In [77]:
df=pd.concat([df, df_high, df_medium, df_low], ignore_index=True)
df=df.sample(frac=1, random_state=42).reset_index(drop=True)

df["label"].value_counts()

label
Low Risk       21
High Risk      13
Medium Risk    10
Name: count, dtype: int64

convert dataframe to csv file 

In [78]:
df.to_csv("../data/cuad_risk_dataset_final.csv", index=False)


Cleaning of dataset 
Remove duplicates columns 


In [79]:
df.drop_duplicates(inplace=True)
df.dropna(inplace=True)

df.shape

(44, 2)

Text Cleaning

In [80]:
def clean_text(text):
    text=text.lower()
    ##Remove everything except letters and spaces
    text=re.sub(r'[^a-z\s]', '', text)
    ###Remove extra spaces
    text=re.sub(r'\s+', ' ', text)
    ###Trim leading and trailing spaces
    return text.strip()

df["clean_text"]=df["text"].apply(clean_text)
df.head()

,text,label,clean_text
0,This document outlines general service informa...,Low Risk,this document outlines general service informa...
1,Payment must be made within thirty days of inv...,Medium Risk,payment must be made within thirty days of inv...
2,Confidential information shall not be disclose...,Medium Risk,confidential information shall not be disclose...
3,The headings are for convenience only.,Low Risk,the headings are for convenience only
4,This agreement defines the terms and condition...,Low Risk,this agreement defines the terms and condition...


Label Encoding

In [81]:
##converts labels to numeric values
le=LabelEncoder()
###look unique labels and transform them to numeric values
df["label_encoded"]=le.fit_transform(df["label"])
le.classes_

array(['High Risk', 'Low Risk', 'Medium Risk'], dtype=object)

Train-Test-Split

In [82]:
X=df["clean_text"]
y=df["label_encoded"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y)   ###stratify to maintain class distribution in train and test sets

TF-Idf (feature extraction) text data--> numeric features

In [83]:
tfidf=TfidfVectorizer(
    max_features=12000,
    ngram_range=(1,3),
    min_df=2,
    max_df=0.85,
    stop_words="english",
    sublinear_tf=True
)

X_train_vec=tfidf.fit_transform(X_train)
X_test_vec=tfidf.transform(X_test)

Model Training

In [84]:
model=LinearSVC(class_weight="balanced", random_state=42)
model.fit(X_train_vec, y_train)

,penalty,'l2'
,loss,'squared_hinge'
,dual,'auto'
,tol,0.0001
,C,1.0
,multi_class,'ovr'
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,verbose,0
,random_state,42


Evalution

In [85]:
y_pred=model.predict(X_test_vec)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=le.classes_))

Accuracy: 0.7777777777777778
              precision    recall  f1-score   support

   High Risk       1.00      0.67      0.80         3
    Low Risk       0.80      1.00      0.89         4
 Medium Risk       0.50      0.50      0.50         2

    accuracy                           0.78         9
   macro avg       0.77      0.72      0.73         9
weighted avg       0.80      0.78      0.77         9



In [86]:
import pickle

pickle.dump(model, open("../artifacts/risk_model.pkl", "wb"))
pickle.dump(tfidf, open("../artifacts/tfidf.pkl", "wb"))
pickle.dump(le, open("../artifacts/label_encoder.pkl", "wb"))
